# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

C:\Users\tacho\anaconda31\envs\tinyml-arduino\python.exe


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df.drop(columns=["Class"]).values   # feature matrix: (n_samples, num_features)
y = df["Class"].values                 

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))

X shape: (178, 13)
y shape: (178,)


Unique labels: [0 1 2]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (124, 13)
X_test shape: (54, 13)
y_train shape: (124,)
y_test shape: (54,)


In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled mean (approx 0):", np.round(X_train_scaled.mean(axis=0), 3))
print("X_train_scaled std  (approx 1):", np.round(X_train_scaled.std(axis=0), 3))

X_train_scaled mean (approx 0): [ 0. -0.  0. -0.  0.  0.  0. -0.  0.  0.  0.  0.  0.]


X_train_scaled std  (approx 1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

y_train_cat shape: (124, 3)
y_test_cat shape: (54, 3)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax'),
])

model.summary()

Model: "sequential"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense (Dense)               (None, 64)                896       


 dense_1 (Dense)             (None, 32)                2080      


 dense_2 (Dense)             (None, 3)                 99        


Total params: 3075 (12.01 KB)


Trainable params: 3075 (12.01 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
)

Epoch 1/20



 1/13 [=>............................] - ETA: 23s - loss: 1.3429 - accuracy: 0.2500


 6/13 [============>.................] - ETA: 0s - loss: 1.1794 - accuracy: 0.2917 


11/13 [========================>.....] - ETA: 0s - loss: 1.1117 - accuracy: 0.3636


13/13 [==============================] - 3s 62ms/step - loss: 1.0888 - accuracy: 0.3838 - val_loss: 0.9023 - val_accuracy: 0.5600


...


 1/13 [=>............................] - ETA: 0s - loss: 0.0601 - accuracy: 1.0000


13/13 [==============================] - 0s 11ms/step - loss: 0.0224 - accuracy: 1.0000 - val_loss: 0.0446 - val_accuracy: 1.0000


Epoch 20/20



 1/13 [=>............................] - ETA: 0s - loss: 0.0328 - accuracy: 1.0000


13/13 [==============================] - 0s 11ms/step - loss: 0.0189 - accuracy: 1.0000 - val_loss: 0.0409 - val_accuracy: 1.0000


In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print("Test accuracy:", test_acc)

y_true = np.argmax(y_test_cat, axis=1)
y_pred = np.argmax(model.predict(X_test_scaled), axis=1)

print("\nClassification report:\n", classification_report(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Test accuracy: 1.0



1/2 [==============>...............] - ETA: 0s


2/2 [==============================] - 0s 5ms/step



Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
 [[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

import os


def file_size_kb(filename):
    """Return the size of a file in kilobytes."""
    return os.path.getsize(filename) / 1024.0


# Convert the trained Keras model to a TFLite model.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Base TFLite model size: {file_size_kb('model_base.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpslkqha5r\assets


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpslkqha5r\assets


Base TFLite model size: 14.07 KB


## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    y_pred = []
    for i in range(len(X_test)):
        x = X_test[i:i + 1].astype(np.float32)

        # Quantize the input if the model expects an integer input.
        if input_details['dtype'] in (np.int8, np.uint8):
            in_scale, in_zero_point = input_details['quantization']
            x = np.round(x / in_scale + in_zero_point).astype(input_details['dtype'])

        interpreter.set_tensor(input_details['index'], x)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])[0]

        # Dequantize the output if the model produces an integer output.
        if output_details['dtype'] in (np.int8, np.uint8):
            out_scale, out_zero_point = output_details['quantization']
            output = (output.astype(np.float32) - out_zero_point) * out_scale

        y_pred.append(np.argmax(output))

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    print(f"{quant_type.upper()} accuracy:", np.mean(y_pred == y_true))
    print("\nClassification report:\n", classification_report(y_true, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpy2qd62ad\assets


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpy2qd62ad\assets


C:\Users\tacho\anaconda31\envs\tinyml-arduino\lib\site-packages\tensorflow\lite\python\convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 5.74 KB
INT8 accuracy: 1.0

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
 [[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmp29nlbz3k\assets


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmp29nlbz3k\assets



FLOAT16 TFLite model size: 8.95 KB
FLOAT16 accuracy: 1.0

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
 [[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpr2iq3ix6\assets


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpr2iq3ix6\assets



DYNAMIC TFLite model size: 8.17 KB
DYNAMIC accuracy: 1.0

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
 [[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

prune_batch_size = 8
prune_epochs = 10
num_train_samples = X_train_scaled.shape[0]
end_step = int(np.ceil(num_train_samples / prune_batch_size) * prune_epochs)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step,
)

print("Total training steps (end_step):", end_step)

Total training steps (end_step): 160


In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

pruned_model = Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    prune_low_magnitude(Dense(64, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(32, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(3, activation='softmax'), pruning_schedule=pruning_schedule),
])

pruned_model.summary()

Model: "sequential_1"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 prune_low_magnitude_dense_  (None, 64)                1730      


 3 (PruneLowMagnitude)                                           


...

 5 (PruneLowMagnitude)                                           


Total params: 6057 (23.67 KB)


Trainable params: 3075 (12.01 KB)


Non-trainable params: 2982 (11.66 KB)


_________________________________________________________________


In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_model.fit(
    X_train_scaled, y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks,
)

Epoch 1/10



 1/13 [=>............................] - ETA: 1:11 - loss: 1.1314 - accuracy: 0.3750


10/13 [======================>.......] - ETA: 0s - loss: 1.0760 - accuracy: 0.4250  


13/13 [==============================] - 7s 55ms/step - loss: 1.0597 - accuracy: 0.4343 - val_loss: 0.9717 - val_accuracy: 0.5600


Epoch 2/10


...


13/13 [==============================] - 0s 14ms/step - loss: 0.4541 - accuracy: 0.9293 - val_loss: 0.3735 - val_accuracy: 0.9200


Epoch 10/10



 1/13 [=>............................] - ETA: 0s - loss: 0.3244 - accuracy: 1.0000


12/13 [==========================>...] - ETA: 0s - loss: 0.3751 - accuracy: 0.9583


13/13 [==============================] - 0s 14ms/step - loss: 0.3715 - accuracy: 0.9596 - val_loss: 0.3161 - val_accuracy: 0.9600


In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

final_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(final_pruned_model)
tflite_pruned_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned_model)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpv098431a\assets


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpv098431a\assets


Pruned TFLite model size: 14.09 KB


In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_true = np.argmax(y_test_cat, axis=1)
y_pred = np.argmax(final_pruned_model.predict(X_test_scaled), axis=1)

print("Pruned model accuracy:", np.mean(y_pred == y_true))
print("\nClassification report:\n", classification_report(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


1/2 [==============>...............] - ETA: 0s


2/2 [==============================] - 0s 7ms/step


Pruned model accuracy: 0.9814814814814815

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       0.95      1.00      0.98        21
           2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  1  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax'),
])

student_model.summary()

Model: "sequential_2"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense_6 (Dense)             (None, 32)                448       


 dense_7 (Dense)             (None, 16)                528       


 dense_8 (Dense)             (None, 3)                 51        


Total params: 1027 (4.01 KB)


Trainable params: 1027 (4.01 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train_scaled)

print("Teacher soft labels shape:", teacher_preds_soft.shape)
print("Example soft label:", teacher_preds_soft[0])


1/4 [======>.......................] - ETA: 0s


4/4 [==============================] - 0s 6ms/step


Teacher soft labels shape: (124, 3)
Example soft label: [0.0019508  0.02744631 0.97060287]


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)
print("Combined label shape:", y_train_combined.shape)

alpha = 0.5


def distillation_loss(y_true_combined, y_pred):

    # Separate the hard one-hot labels from the soft teacher labels.
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    # Loss against the ground-truth hard labels and against the teacher soft labels.
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    # Weighted combination of the two losses.
    return alpha * hard_loss + (1 - alpha) * soft_loss

Combined label shape: (124, 6)


In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
)

student_model.fit(
    X_train_scaled, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
)

Epoch 1/10



 1/13 [=>............................] - ETA: 24s - loss: 1.2043


13/13 [==============================] - ETA: 0s - loss: 1.1651 


13/13 [==============================] - 3s 54ms/step - loss: 1.1651 - val_loss: 0.9727


Epoch 2/10


...


 1/13 [=>............................] - ETA: 0s - loss: 0.2626


13/13 [==============================] - 0s 12ms/step - loss: 0.3189 - val_loss: 0.2802


Epoch 10/10



 1/13 [=>............................] - ETA: 0s - loss: 0.3391


13/13 [==============================] - 0s 13ms/step - loss: 0.2755 - val_loss: 0.2552


In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_student_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_student_model)

print(f"KD student TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpqcrk_1pf\assets


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmpqcrk_1pf\assets


KD student TFLite model size: 6.10 KB


In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_true = np.argmax(y_test_cat, axis=1)
y_pred = np.argmax(student_model.predict(X_test_scaled), axis=1)

print("Student model accuracy:", np.mean(y_pred == y_true))
print("\nClassification report:\n", classification_report(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


1/2 [==============>...............] - ETA: 0s


2/2 [==============================] - 0s 7ms/step


Student model accuracy: 0.9814814814814815

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

The int8 model had the lowest kb of SRAM memory while giving out very high performance. 

2. **Propose a strategy** that combines or enhances techniques learned so far.

Having sometype of int8 and knowledge distillation to make the model more efficient with fewer memory.

5. **Implement** your proposed solution.

I have the implementation down with the code before

6. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

7. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [ ]:
# <-- (if needed) Enter your code here <--#
#
# Combine techniques instead of using them in isolation. This stacks a smaller
# network with 4x-smaller int8 weights to push the size below any single
# technique from other parts for less memory usage and faster inference.

quantize_and_evaluate(
    student_model, X_test_scaled, y_test_cat, 'int8', 'model_kd_int8.tflite'
)

# Size comparison across every model produced in this problem.
model_files = [
    ("Base (float32)", "model_base.tflite"),
    ("Dynamic range", "model_dynamic.tflite"),
    ("Float16", "model_float16.tflite"),
    ("Int8", "model_int8.tflite"),
    ("Pruned", "model_pruned.tflite"),
    ("KD student", "model_kd.tflite"),
    ("KD + int8 (combined)", "model_kd_int8.tflite"),
]

print("\nModel size summary")
print("-" * 40)
for name, path in model_files:
    if os.path.exists(path):
        print(f"{name:<24} {file_size_kb(path):>8.2f} KB")

INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmp71bo2_1k\assets


INFO:tensorflow:Assets written to: C:\Users\tacho\AppData\Local\Temp\tmp71bo2_1k\assets


C:\Users\tacho\anaconda31\envs\tinyml-arduino\lib\site-packages\tensorflow\lite\python\convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 3.62 KB
INT8 accuracy: 0.9814814814814815

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]

Model size summary
----------------------------------------
Base (float32)              14.07 KB
Dynamic range                8.17 KB
Float16                      8.95 KB
Int8                         5.74 KB
Pruned                      14.09 KB
KD student                   6.10 KB
KD + int8 (combined)         3.62 KB


# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
